In [ ]:
%%time
# references
# S. Bravy and A. Kitaev, PRA71(2005)022316

# final version
# test (initial state with a parameter) + (dephasing circuit)
# backend = AerSimulator() and SamplerV2()

# Information of the local environment

# python: 3.11.13
# jupyter notebook: 7.4.4
# qiskit: 2.5.2              
# qiskit-aer: 0.17.2
# qiskit-experiments: 0.14.2
# qiskit-ibm-experiment: 0.4.8
# qiskit-ibm-runtime: 0.49.0
import qiskit

import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit import Parameter
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import plot_histogram, plot_state_city
from qiskit.quantum_info import SparsePauliOp, Operator
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Session, QiskitRuntimeService, SamplerV2 as Sampler

# module
from five_qubits_code_T_type_distillation import epsilon, \
                                                func_p_out, \
                                                func_error_out, \
                                                generate_rho_initial_5_qubits, \
                                                initialize_data_qubits, \
                                                decimal_to_base_3_number, \
                                                t_type_dephasing_operation_5_qubits, \
                                                initialize_state_ancilla_qubits, \
                                                syndrome_4_ancillas_no_reset_decoding, \
                                                generate_state_t0_t1, \
                                                calculate_accept_rate_and_error_rate

# mwc

NUM_PHY_Q = 9
NUM_Q = 5
NUM_ANC = 4

# state preparation
# in this example code, epsilon ( = epsilon_ini) which is a parameter in the initial mixed state to be distilled. is defined by 
# epsilon = (1 - np.cos( delta )) / np.sqrt(6), where delta =  PHI - phi
# 0 < delta < PHI (0 < phi < PHI)
# phi = PHI corresponds to delta = 0 ( epsilon = 0 )

phi = Parameter("φ")
phi_list = [i * 0.05 for i in range(16)]

# These parameters are also given in the module "five_qubits_code_T_type_distillation"

THETA = np.arctan(np.sqrt(2))
PHI = np.pi / 4

# backend
aer_backend = AerSimulator(max_parallel_threads=4, max_parallel_experiments=4, max_parallel_shots=1)

SHOTS = 10000
NUM_CIRC = 243 # dephasing is exact if there is no error.    

accept_counts = [0. for phi in phi_list]
accept_counts_check = [0. for phi in phi_list]
error_counts = [0. for phi in phi_list]

sampler = Sampler(mode=aer_backend)
pm = generate_preset_pass_manager(optimization_level=1, backend=aer_backend)
params = phi_list

for i in range(NUM_CIRC):

    qr_data = QuantumRegister(NUM_Q, name='q')
    qr_anc = QuantumRegister(NUM_ANC, name='anc')
    circ = QuantumCircuit(qr_data, qr_anc)
    
    initialize_state_ancilla_qubits(qc=circ, q_anc=qr_anc)
    initialize_data_qubits(qc=circ, q_data=qr_data, phi=phi)
    
    t_type_dephasing_operation_5_qubits(qc=circ, q_data=qr_data, input_num=i)
    syndrome_4_ancillas_no_reset_decoding(qc=circ, q_data=qr_data, q_anc=qr_anc, phi=phi)
    
    circ.measure_all()

    # test
    if i == 0:
        display(circ.draw("mpl", fold=-1))
                
    else:      
        pass
    
    transpiled_circ = pm.run(circ)
    pub = (transpiled_circ, params)
    
    job = sampler.run([pub], shots=SHOTS)
    result = job.result()
    
    for idx, angle in enumerate(phi_list):
        counts_all = result[0].data.meas[idx].get_counts()
        counts_all_keys_list = counts_all.keys()
        counts_all_values_list = counts_all.values()

        for kdx, all_string in enumerate(counts_all_keys_list):

            if all_string[:4] == "0000":
                
                accept_counts[idx] += counts_all[all_string]
    
                if all_string[4:] == "00000":
                    
                    accept_counts_check[idx] += counts_all[all_string]
                    
                elif all_string[4:] == "00001":
                    
                    accept_counts_check[idx] += counts_all[all_string]
                    error_counts[idx] += counts_all[all_string]

                else:
                    
                    accept_counts_check[idx] += 0
                    error_counts[idx] += 0                    

            else:
                accept_counts[idx] += 0

        # Confirm that accept_counts is equal to accept_counts_check
        assert accept_counts[idx] == accept_counts_check[idx]

        assert len(accept_counts) == len(error_counts)

accept_rates, error_rates = calculate_accept_rate_and_error_rate(shots=SHOTS, reps=NUM_CIRC, \
                                                  accept_counts_list=accept_counts, error_counts_list=error_counts)
print(accept_rates)

# curve fit and visualization

import matplotlib.pyplot as plt
import scipy 
from scipy.optimize import curve_fit

# mwc for P_out

# epsilon = (1 - np.cos( delta )) / np.sqrt(6), where delta =  PHI - phi

xdata = phi_list
epsilondata = [epsilon(phi) for phi in phi_list]
th_accept_data = [func_p_out(epsi) for epsi in epsilondata]

assert len(th_accept_data) == len(xdata)

ydata = accept_rates

# udata = accept_rates_check

figure, ax = plt.subplots()

ax.plot(xdata, th_accept_data, 'x-', label= "theory")
ax.plot(xdata, ydata, 'o', label="data")
# ax.plot(xdata, udata, '+', label="data_check")

plt.xlabel("phi_in")
plt.ylabel("P_accept")
plt.legend()
plt.show()

step2 = False
if step2 = True:
    
    # mwc for Error_out
        
    xdata = phi_list
    th_e_out_data = [func_error_out(epsi / (1 - epsi)) for epsi in epsilondata]
       
    assert len(th_e_out_data) == len(xdata)
    
    zdata = error_rates
    
    figure, ax = plt.subplots()
    
    ax.plot(xdata, th_e_out_data, 'x-', label="theory")
    ax.plot(xdata, zdata, 'o', label="data") 
    
    plt.xlabel("phi_in")
    plt.ylabel("epsilon_out")
    plt.legend()
    plt.show()